In [1]:
from typing import Any, TypedDict

from langchain.chat_models import init_chat_model
from langchain_core.messages import AnyMessage
from langchain_core.messages.utils import count_tokens_approximately
from langgraph.graph import StateGraph, START, MessagesState
from langgraph.checkpoint.memory import InMemorySaver
from langmem.short_term import SummarizationNode

import dotenv

dotenv.load_dotenv()

True

In [2]:
from langchain_google_genai import ChatGoogleGenerativeAI


model=ChatGoogleGenerativeAI(model="gemini-2.0-flash")
summarization_model = model.bind(max_tokens=128)

In [3]:
class State(MessagesState):
    context: dict[str, Any]  

class LLMInputState(TypedDict):  
    summarized_messages: list[AnyMessage]
    context: dict[str, Any]

summarization_node = SummarizationNode(
    token_counter=count_tokens_approximately,
    model=summarization_model,
    max_tokens=256,
    max_tokens_before_summary=256,
    max_summary_tokens=128,
)

In [4]:
def call_model(state: LLMInputState):  
    response = model.invoke(state["summarized_messages"])
    return {"messages": [response]}

In [9]:
checkpointer = InMemorySaver()
builder = StateGraph(State)
builder.add_node(call_model)
builder.add_node("summarize", summarization_node)
builder.add_edge(START, "summarize")
builder.add_edge("summarize", "call_model")
graph = builder.compile(checkpointer=checkpointer)

In [10]:
# Invoke the graph
config = {"configurable": {"thread_id": "1"}}
graph.invoke({"messages": "hi, my name is bob"}, config)
graph.invoke({"messages": "write a short poem about cats"}, config)
graph.invoke({"messages": "now do the same but for dogs"}, config)
final_response = graph.invoke({"messages": "what's my name?"}, config)

In [11]:
final_response["messages"][-1].pretty_print()
print("\nSummary:", final_response["context"]["running_summary"].summary)

================================== Ai Message ==================================

Your name is Bob. You told me that at the beginning of our conversation. 😉


KeyError: 'context'

### lowering max_token=20

In [21]:
summarization_node = SummarizationNode(
    token_counter=count_tokens_approximately,
    model=summarization_model,
    max_tokens=128,
    max_tokens_before_summary=10,
    max_summary_tokens=5,
)

In [22]:
checkpointer = InMemorySaver()
builder = StateGraph(State)
builder.add_node(call_model)
builder.add_node("summarize", summarization_node)
builder.add_edge(START, "summarize")
builder.add_edge("summarize", "call_model")
graph = builder.compile(checkpointer=checkpointer)

In [23]:
# Invoke the graph
config = {"configurable": {"thread_id": "1"}}
graph.invoke({"messages": "hi, my name is bob"}, config)
graph.invoke({"messages": "write a short poem about cats"}, config)
graph.invoke({"messages": "now do the same but for dogs"}, config)
final_response = graph.invoke({"messages": "what's my name?"}, config)

In [24]:
final_response["messages"][-1].pretty_print()
print("\nSummary:", final_response["context"]["running_summary"].summary)

================================== Ai Message ==================================

Your name is Bob. You introduced yourself at the beginning of our conversation.

Summary: This is a summary of the conversation so far: The user, Bob, introduced himself. I responded politely and offered assistance. The user then requested a short poem about cats, and I provided a four-stanza poem fulfilling that request.


In [25]:
graph.get_state(config).values

{'messages': [HumanMessage(content='hi, my name is bob', additional_kwargs={}, response_metadata={}, id='bc319d1a-3eba-4039-a78f-81b9bbb518fb'),
  AIMessage(content="Hi Bob! It's nice to meet you. How can I help you today?", additional_kwargs={}, response_metadata={'prompt_feedback': {'block_reason': 0, 'safety_ratings': []}, 'finish_reason': 'STOP', 'model_name': 'gemini-2.0-flash', 'safety_ratings': []}, id='run--e57a7cff-0732-4d05-81fa-8c331b90c6c7-0', usage_metadata={'input_tokens': 6, 'output_tokens': 19, 'total_tokens': 25, 'input_token_details': {'cache_read': 0}}),
  HumanMessage(content='write a short poem about cats', additional_kwargs={}, response_metadata={}, id='a8c28ee1-178d-4dc9-bf3b-22457af9164d'),
  AIMessage(content="With eyes of jade and coat of night,\nA feline shadow, soft and light.\nA purring rumble, low and deep,\nWhile secrets in their silence sleep.\n\nA playful pounce, a graceful leap,\nA hunter's instinct, buried deep.\nA furry friend, a regal grace,\nA cat'

In [ ]:
graph.get_state(config).values

{'messages': [HumanMessage(content='hi, my name is bob', additional_kwargs={}, response_metadata={}, id='bc319d1a-3eba-4039-a78f-81b9bbb518fb'),
  AIMessage(content="Hi Bob! It's nice to meet you. How can I help you today?", additional_kwargs={}, response_metadata={'prompt_feedback': {'block_reason': 0, 'safety_ratings': []}, 'finish_reason': 'STOP', 'model_name': 'gemini-2.0-flash', 'safety_ratings': []}, id='run--e57a7cff-0732-4d05-81fa-8c331b90c6c7-0', usage_metadata={'input_tokens': 6, 'output_tokens': 19, 'total_tokens': 25, 'input_token_details': {'cache_read': 0}}),
  HumanMessage(content='write a short poem about cats', additional_kwargs={}, response_metadata={}, id='a8c28ee1-178d-4dc9-bf3b-22457af9164d'),
  AIMessage(content="With eyes of jade and coat of night,\nA feline shadow, soft and light.\nA purring rumble, low and deep,\nWhile secrets in their silence sleep.\n\nA playful pounce, a graceful leap,\nA hunter's instinct, buried deep.\nA furry friend, a regal grace,\nA cat'

In [ ]:
graph.get_state(config).values

{'messages': [HumanMessage(content='hi, my name is bob', additional_kwargs={}, response_metadata={}, id='bc319d1a-3eba-4039-a78f-81b9bbb518fb'),
  AIMessage(content="Hi Bob! It's nice to meet you. How can I help you today?", additional_kwargs={}, response_metadata={'prompt_feedback': {'block_reason': 0, 'safety_ratings': []}, 'finish_reason': 'STOP', 'model_name': 'gemini-2.0-flash', 'safety_ratings': []}, id='run--e57a7cff-0732-4d05-81fa-8c331b90c6c7-0', usage_metadata={'input_tokens': 6, 'output_tokens': 19, 'total_tokens': 25, 'input_token_details': {'cache_read': 0}}),
  HumanMessage(content='write a short poem about cats', additional_kwargs={}, response_metadata={}, id='a8c28ee1-178d-4dc9-bf3b-22457af9164d'),
  AIMessage(content="With eyes of jade and coat of night,\nA feline shadow, soft and light.\nA purring rumble, low and deep,\nWhile secrets in their silence sleep.\n\nA playful pounce, a graceful leap,\nA hunter's instinct, buried deep.\nA furry friend, a regal grace,\nA cat'

In [ ]:
graph.get_state(config).values

{'messages': [HumanMessage(content='hi, my name is bob', additional_kwargs={}, response_metadata={}, id='bc319d1a-3eba-4039-a78f-81b9bbb518fb'),
  AIMessage(content="Hi Bob! It's nice to meet you. How can I help you today?", additional_kwargs={}, response_metadata={'prompt_feedback': {'block_reason': 0, 'safety_ratings': []}, 'finish_reason': 'STOP', 'model_name': 'gemini-2.0-flash', 'safety_ratings': []}, id='run--e57a7cff-0732-4d05-81fa-8c331b90c6c7-0', usage_metadata={'input_tokens': 6, 'output_tokens': 19, 'total_tokens': 25, 'input_token_details': {'cache_read': 0}}),
  HumanMessage(content='write a short poem about cats', additional_kwargs={}, response_metadata={}, id='a8c28ee1-178d-4dc9-bf3b-22457af9164d'),
  AIMessage(content="With eyes of jade and coat of night,\nA feline shadow, soft and light.\nA purring rumble, low and deep,\nWhile secrets in their silence sleep.\n\nA playful pounce, a graceful leap,\nA hunter's instinct, buried deep.\nA furry friend, a regal grace,\nA cat'